# VICTRE-Paired v11 — generation (Colab)

Runs `dataset_generate.py` over the VICTRE collection and validates the result with `dataset_test.py`.

| | cell | when |
|---|---|---|
| **A** | setup | in every session, first |
| **B** | settings | in every session, after A |
| **C** | smoke run — 8 phantoms | **once**, at the start |
| **D** | pilot run — 100 phantoms | **once**, after C |
| **E** | full generation | resumes where it stopped; run it as often as needed |
| **F** | validation | after generation |
| **G** | manifest | after F |
| **H** | status | at any time |

**Several sessions:** cell E can run in 2–3 tabs at once. Set `SESSION` and `N_SESSIONS`; the tabs see each
other through Drive and never generate the same phantom twice.

## A · Setup

In [ ]:
# GPU SART is required: ASTRA's 3D algorithms have no CPU path.
import subprocess, sys, os
out = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert out, "NO GPU -> Runtime > Change runtime type > GPU"
print("GPU:", out)

!pip -q install astra-toolbox pydicom tqdm 2>&1 | tail -2
import astra
print("ASTRA", astra.__version__, "| CUDA:", astra.astra.use_cuda())
assert astra.astra.use_cuda(), "ASTRA has no CUDA build"

from google.colab import drive
drive.mount('/content/drive')

## B · Settings

In [ ]:
from pathlib import Path
import os, shutil, time, json, glob

# --------------------------------------------------------------------- paths
NB_DIR    = Path("/content/drive/MyDrive/victre_paired/code")   # dataset_generate.py + dataset_test.py
                                                        # + dbt_2026v1_manifest.csv are uploaded here
DRIVE_OUT = Path("/content/drive/MyDrive/New_DBT/VICTRE-PAIRED-v11")   # persistent output
LOCAL_OUT = Path("/content/v10_out")                    # VM disk: fast, temporary
CACHE     = Path("/content/victre_cache")               # download cache, cleared after each phantom

# ----------------------------------------------------------------- session/parallel
SESSION    = 0      # index of this tab (0..N_SESSIONS-1)
N_SESSIONS = 1      # number of tabs running at once
BATCH      = 25     # move to Drive every this many phantoms

for d in (LOCAL_OUT, CACHE, DRIVE_OUT):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(NB_DIR)
assert (NB_DIR/"dataset_generate.py").exists(), f"scripts not under {NB_DIR}"
assert (NB_DIR/"dbt_2026v1_manifest.csv").exists(), "the frozen split manifest is missing"

SPLITS = ("train","val","test")

def drive_done():
    """Ids of the phantoms finished on Drive. One-level glob, no tree walk."""
    return {p.stem for s in SPLITS for p in (DRIVE_OUT/s).glob("*.npz")}

def move_to_drive():
    """Moves the npz files finished locally to Drive. If FUSE drops, remounts and continues."""
    moved = 0
    for s in SPLITS:
        src = LOCAL_OUT/s
        if not src.is_dir(): continue
        (DRIVE_OUT/s).mkdir(parents=True, exist_ok=True)
        for f in sorted(src.glob("*.npz")):
            if ".partial" in f.name: continue   # interrupted while writing; the generator rewrites it
            for attempt in range(3):
                try:
                    shutil.move(str(f), str(DRIVE_OUT/s/f.name)); moved += 1; break
                except OSError as e:
                    print(f"  ! {f.name}: {e} (attempt {attempt+1})")
                    time.sleep(10)
                    try: drive.mount('/content/drive', force_remount=True)
                    except Exception: pass
    for extra in ("config.json","progress.json","manifest.csv"):
        p = LOCAL_OUT/extra
        if p.exists(): shutil.copy(str(p), str(DRIVE_OUT/extra))
    return moved

print(f"finished on Drive: {len(drive_done())} phantoms")

## C · Smoke run — 8 phantoms

The first 8 phantoms are **development-set phantoms (BIAI Lab DBT 2026v1)**, one per density and lesion status.

`proj_full`, `proj_valid`, `sart_full`, `breast_mask`, `lesions` **do not depend on the dose**: they must
match the development release up to floating-point differences. A large difference means the pipeline is
broken: **stop and find the cause.**

In [ ]:
!python dataset_generate.py --cache /content/smoke_cache --output /content/smoke --limit 8 --keep-cache

import numpy as np
from pathlib import Path
fs = sorted(Path("/content/smoke").rglob("*.npz"))
print(f"\n{len(fs)} files written\n")
d = np.load(fs[0])
print("keys:", sorted(d.files))
for k in ("proj_full","proj_50","proj_25","proj_10","sart_full","breast_mask",
          "victre_fbp","victre_fbp_present","victre_fbp_scale","victre_fbp_saturated"):
    if k in d.files:
        a = d[k]
        print(f"  {k:<22} {str(a.dtype):<8} {a.shape}")
    else:
        print(f"  {k:<22} MISSING")

### C2 · Compare with the development release

Needs the `.npz` files of **the same phantoms** from the development release `dataset_biailab_DBT_2026v1`.
Write their location into `REF_DIR`. Without them, skip this cell; the port then remains **unverified**.

In [ ]:
REF_DIR = Path("/content/drive/MyDrive/dataset_biailab_DBT_2026v1")   # development release

if not REF_DIR.exists():
    print("DEVELOPMENT RELEASE NOT FOUND -> comparison skipped, port NOT VERIFIED")
else:
    SAME = ("proj_full","proj_valid","sart_full","breast_mask","lesions")   # independent of the dose
    for f in sorted(Path("/content/smoke").rglob("*.npz")):
        ref = next((r for r in REF_DIR.rglob(f.name)), None)
        if ref is None:
            print(f"{f.stem}: not in the development release, skipped"); continue
        a, b = np.load(f), np.load(ref)
        print(f"\n{f.stem}")
        for k in SAME:
            if k not in b.files: print(f"  {k:<14} not in the development release"); continue
            x, y = a[k].astype(np.float64), b[k].astype(np.float64)
            if x.shape != y.shape:
                print(f"  {k:<14} SHAPE DIFFERS {x.shape} vs {y.shape}  <-- STOP"); continue
            rel = np.abs(x-y).max() / max(np.abs(y).max(), 1e-12)
            flag = "" if rel < 1e-3 else ("  <-- LARGE DIFFERENCE, STOP" if rel > 1e-2 else "  <-- check")
            print(f"  {k:<14} max relative difference {rel:.2e}{flag}")

### C3 · Smoke validation — no development release needed

The **alignment** of `victre_fbp` (the one place the addition can break silently), the dose order and the units.
The alignment correlation is measured inside the breast: both volumes reconstruct the same projections, so it
must be clearly positive. **Below 0.30 = wrong grid mapping, stop.**

In [ ]:
import numpy as np
from pathlib import Path

fs = sorted(Path("/content/smoke").rglob("*.npz"))
print(f"{'phantom':<13}{'density':<11}{'slices':>6}{'vfbp':>6}{'corr':>8}{'satur':>9}"
      f"{'GT med':>9}{'sig50':>8}{'sig25':>8}{'sig10':>8}")
print("-"*88)

bad = []
for f in fs:
    d = np.load(f)
    dn, S = str(d["density"]), d["sart_full"].shape[0]
    m, gt = d["breast_mask"], d["sart_full"].astype(np.float64)
    gt_med = float(np.median(gt[m]))                      # mm^-1, ~0.04 expected

    # alignment of the second reference
    present = bool(d["victre_fbp_present"])
    c = float("nan")
    if present:
        a = d["victre_fbp"][m].astype(np.float64)
        if a.std() > 0:
            c = float(np.corrcoef(a, gt[m])[0, 1])
            if c < 0.30: bad.append((f.stem, "alignment", c))

    # dose order: the noise must GROW as the dose falls
    sig = {}
    for k in ("50", "25", "10"):
        x = d[f"proj_{k}"].astype(np.float64); y = d["proj_full"].astype(np.float64)
        fp = y > 0
        sig[k] = float((x - y)[fp].std())
    if not (sig["50"] < sig["25"] < sig["10"]):
        bad.append((f.stem, "dose order", tuple(sig.values())))

    print(f"{f.stem:<13}{dn:<11}{S:>6}{str(present):>6}{c:>8.3f}"
          f"{float(d['victre_fbp_saturated']):>9.2e}{gt_med:>9.4f}"
          f"{sig['50']:>8.4f}{sig['25']:>8.4f}{sig['10']:>8.4f}")

print("-"*88)
print(f"with victre_fbp: {sum(bool(np.load(f)['victre_fbp_present']) for f in fs)} / {len(fs)}")
print("PROBLEMS:", bad if bad else "NONE")

# by eye: the same slice, both references side by side
import matplotlib.pyplot as plt
f = next(f for f in fs if bool(np.load(f)["victre_fbp_present"]))
d = np.load(f); z = d["sart_full"].shape[0] // 2
m = d["breast_mask"][z]
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, (img, t) in zip(ax, [(d["sart_full"][z], "sart_full (GT)"),
                            (d["victre_fbp"][z].astype(np.float32), "victre_fbp (second reference)"),
                            (m.astype(float), "breast_mask")]):
    img = img.astype(np.float32)
    if m.any() and t != "breast_mask":
        lo, hi = np.percentile(img[m], [2, 98]); a.imshow(img, cmap="gray", vmin=lo, vmax=hi)
    else:
        a.imshow(img, cmap="gray")
    a.set_title(f"{f.stem} {str(d['density'])}: {t}", fontsize=9); a.axis("off")
plt.tight_layout(); plt.show()

## D · Pilot run — 100 phantoms

**Not to be skipped.** The pipeline had never run at the scale of 2,993 phantoms; the development release reports
"at most 3 attempts" on 384. Measured here: real download speed, failure rate, time per phantom, output size,
and whether the second reference works.

In [ ]:
import time
from pathlib import Path
SCAN = Path("/content/pilot")   # D2/D3/D4 read this folder
t0 = time.time()
!python dataset_generate.py --cache /content/pilot_cache --output /content/pilot --limit 100 --progress-every 10
dt = time.time() - t0

import numpy as np, json
from pathlib import Path
fs = sorted(SCAN.rglob("*.npz"))
mb = sum(f.stat().st_size for f in fs)/1e6
prog = json.loads((SCAN/"progress.json").read_text())

print(f"\n{'='*58}")
print(f"  phantoms            : {len(fs)} / 100   failed: {len(prog.get('failed',{}))}")
print(f"  time                : {dt/60:.1f} min   -> {dt/max(len(fs),1):.1f} s/phantom")
print(f"  estimate for 2993   : {dt/max(len(fs),1)*2993/3600:.1f} hours")
print(f"  size                : {mb/len(fs):.1f} MB/phantom -> {mb/len(fs)*2993/1000:.0f} GB")
vp = [bool(np.load(f)["victre_fbp_present"]) for f in fs]
print(f"  with victre_fbp     : {sum(vp)} / {len(vp)}")
if prog.get("failed"): print("  ERRORS:", prog["failed"])
print(f"{'='*58}")

### D1f · Generate fatty phantoms only — instead of D

**D need not be run again.** The 100-phantom pilot was done and answered its question: 0 failures, 67.8 s/phantom
(56.3 h for 2,993), 31.3 MB/phantom (94 GB), `victre_fbp` in 81/100. Its output was on the VM disk and was lost with
the session.

D3's question concerns **fatty only**: the deviation of dense/hetero/scattered is below 0.3 slice and none reaches 2σ.
So 30 fatty phantoms are generated instead of 100: faster (~45 min) and ~60 masses instead of the 48 available to D2.

No new code: `--done-list` takes every phantom that is not fatty, and the generator produces the rest.

**Not copied to Drive.** If `DELTA[fatty]` changes these files become invalid; the decision is made after D3, with D4.

In [ ]:
# D1f: generate fatty only. NOT copied to Drive -- decision after D3.
import subprocess, sys, time
import numpy as np
from pathlib import Path

sys.path.insert(0, str(NB_DIR))
import dataset_generate as G

SCAN    = Path("/content/fatty_probe")      # D2/D3/D4 read this folder
N_FATTY = 30
FRESH   = True      # True if DELTA changed: the generator skips existing targets,
                    # so unless the old files are deleted the new DELTA is never tried.

if FRESH and SCAN.exists():
    import shutil as _sh
    _sh.rmtree(SCAN)
    print("old", SCAN, "deleted (FRESH=True)")
SCAN.mkdir(parents=True, exist_ok=True)

items = G.select_patients(G.tcia_index(CACHE), G.lesion_index(CACHE))
skip  = [it["patient"] for it in items if it["density"] != "fatty"]
(SCAN/"done.txt").write_text("\n".join(sorted(skip)))
print(f"{len(items)} eligible phantoms, {len(items)-len(skip)} fatty. {N_FATTY} will be generated.\n")

t0 = time.time()
r = subprocess.run([sys.executable, "dataset_generate.py",
                    "--cache", str(CACHE), "--output", str(SCAN),
                    "--done-list", str(SCAN/"done.txt"),
                    "--max-new", str(N_FATTY), "--progress-every", "5"], text=True)
dt = time.time() - t0

fs = sorted(SCAN.rglob("*.npz"))
dn = {str(np.load(f)["density"]) for f in fs}
vp = sum(bool(np.load(f)["victre_fbp_present"]) for f in fs)
print(f"\n{'='*58}")
print(f"  fatty phantoms      : {len(fs)}   exit code: {r.returncode}")
print(f"  time                : {dt/60:.1f} min -> {dt/max(len(fs),1):.0f} s/phantom")
print(f"  densities           : {dn or '-'}   (fatty only expected)")
print(f"  with victre_fbp     : {vp} / {len(fs)}")
print(f"{'='*58}")
assert dn <= {"fatty"}, "non-fatty phantoms were generated, done.txt is wrong"

print(f"\nDELTA[fatty] used = {G.DELTA['fatty']}")
print("Now D2: the offset should be +0.1 +- 0.15. If it is -1.2, the DELTA change")
print("has not reached dataset_generate.py on Drive.")

### D2 · z calibration — against VICTRE's own lesion coordinates

The `.loc` files give the lesion positions **on VICTRE's grid**, an independent reference that depends on no
reconstruction. The true slice of each mass is found from the peak of its contrast profile; parabolic
interpolation gives **sub-slice** resolution.

Measured on the smoke run (4 phantoms): `victre_fbp` at exactly 0, `sart_full` ≈ −1. Here it is checked on n≈400 masses.

In [ ]:
import numpy as np, collections
from pathlib import Path

R  = int(round(2.5/0.34))          # lesion radius, pixels
DZ = np.arange(-3, 4)

def contrast(sl, r, c):
    yy, xx = np.ogrid[:sl.shape[0], :sl.shape[1]]
    d2 = (yy-r)**2 + (xx-c)**2
    core, ring = d2 <= R*R, (d2 > (2*R)**2) & (d2 <= (3*R)**2)
    if core.sum() < 5 or ring.sum() < 20: return np.nan
    return float(sl[core].mean() - sl[ring].mean())

def peak_offset(vol, z, r, c):
    """True slice of the lesion minus the .loc Z. Sub-slice by parabolic interpolation."""
    S = vol.shape[0]
    if not (3 <= z < S-3): return np.nan
    y = np.array([contrast(vol[int(z+d)].astype(np.float64), r, c) for d in DZ])
    if not np.isfinite(y).all(): return np.nan
    k = int(np.argmax(y))
    if k in (0, len(y)-1): return float(DZ[k])          # at the edge, no interpolation
    a, b, cc = y[k-1], y[k], y[k+1]
    den = a - 2*b + cc
    sub = 0.5*(a-cc)/den if abs(den) > 1e-12 else 0.0
    return float(DZ[k] + np.clip(sub, -1, 1))

acc = collections.defaultdict(list)
files = sorted(SCAN.rglob("*.npz"))
for f in files:
    d = np.load(f)
    dn, les = str(d["density"]), d["lesions"]
    has_vf = bool(d["victre_fbp_present"])
    for kind, label in ((1, "mass"), (0, "calcification")):
        rows = les[les[:,4] == kind]
        for s, r, c, _, _ in rows:
            o = peak_offset(d["sart_full"], s, r, c)
            if np.isfinite(o): acc[("sart_full", label, dn)].append(o)
            if has_vf:
                o = peak_offset(d["victre_fbp"], s, r, c)
                if np.isfinite(o): acc[("victre_fbp", label, dn)].append(o)

print(f"{len(files)} phantoms scanned\n")
for label in ("mass", "calcification"):
    print(f"--- {label} ---")
    print(f"{'volume':<12}{'density':<12}{'n':>5}{'offset (slices)':>16}{'std':>8}")
    for vol in ("sart_full", "victre_fbp"):
        for dn in ("dense", "hetero", "scattered", "fatty", None):
            if dn is None:
                v = [x for (vv,ll,_),xs in acc.items() if vv==vol and ll==label for x in xs]
                dn = "ALL"
            else:
                v = acc.get((vol, label, dn), [])
            if len(v) >= 5:
                print(f"{vol:<12}{dn:<12}{len(v):>5}{np.mean(v):>+16.2f}{np.std(v):>8.2f}")
    print()

print("Reading: victre_fbp ~0 and sart_full ~-1 confirm the smoke run.")
print("         Both ~0 means no offset. A difference below 0.3 slice is negligible.")

### D3 · Fatty breakdown — no new reconstruction

D2 points to a single place: in dense/hetero/scattered the deviation is below 0.3 slice (not significant), in fatty
`sart_full` is at **-1.18 +- 0.15** slices (n=48 masses, 8 sigma). Fatty is also:

- the class with the highest `DELTA[fatty] = 20.807` of the four densities (the others
  19.765 / 20.242 / 20.091; the published analytic value is 19.5);
- the only class for which VICTRE's own reconstruction is **mostly missing** on TCIA: in the pilot only 12 of the
  48 masses have `victre_fbp`. In v9, `DELTA[fatty]` was therefore fitted by parallax on this 25 % minority.

This cell makes three separations; none needs new generation, `/content/pilot` is read again (~3 min):

1. **Paired comparison.** Only phantoms with `victre_fbp`, which removes the possibility that the two numbers
   come from two different sets of phantoms.
2. **Selection test.** Is the `sart_full` deviation the same in fatty phantoms with and without `victre_fbp`?
   If not, the problem is the v9 fit set, not the geometry.
3. **Depth test.** Does the deviation depend on the z position of the lesion in the volume?
   - **Constant** (slope ~0) -> an error in the grid origin, i.e. `DELTA[fatty]`.
   - **Sloped** -> a z bias of the reconstruction or a measurement artefact; DELTA is not changed.

**Decision rule.** If 2 shows no difference AND 3 shows slope ~0, set `DELTA[fatty]` to
20.807 - (measured deviation) and regenerate fatty only (~25 %, ~12 h). Otherwise DELTA is not changed and the
deviation is recorded as a known property of the dataset.

> The calcification arm is not used for this decision: its std in D2 is 2.7 slices, the whole of the
> +-3 search window, so the measurement cannot be told apart from noise. Microcalcifications are not
> resolved at 0.34 mm, which is already a stated limitation.

In [ ]:
# D3: fatty breakdown. NO new reconstruction, SCAN is read again.
# D2 must have run in the same session (contrast() and peak_offset() come from there).
import numpy as np, collections
from pathlib import Path

try:
    peak_offset
except NameError:
    raise SystemExit("Run cell D2 first: peak_offset() is not defined.")

recs = []   # (pid, density, zfrac, offset_sart, offset_vfbp, has_vf)
for f in sorted(SCAN.rglob("*.npz")):
    d = np.load(f)
    dn, les = str(d["density"]), d["lesions"]
    S = d["sart_full"].shape[0]
    has_vf = bool(d["victre_fbp_present"])
    vf = d["victre_fbp"] if has_vf else None
    for s, r, c, _, kind in les:
        if int(kind) != 1:                      # masses only
            continue
        o_s = peak_offset(d["sart_full"], s, r, c)
        o_v = peak_offset(vf, s, r, c) if has_vf else np.nan
        recs.append((f.stem, dn, float(s) / S, o_s, o_v, has_vf))

DN = ("dense", "hetero", "scattered", "fatty")
def sem(v):
    return float(np.std(v, ddof=1) / np.sqrt(len(v))) if len(v) > 1 else np.nan

print(f"{len(recs)} masses, {len(set(r[0] for r in recs))} phantoms\n")

# --- 1. paired: the same lesion, two volumes --------------------------------
print("--- 1. paired (only lesions with victre_fbp) ---")
print(f"{'density':<12}{'n':>5}{'sart_full':>12}{'victre_fbp':>12}{'diff':>10}{'sem':>8}")
for dn in DN + (None,):
    p = [(r[3], r[4]) for r in recs
         if (dn is None or r[1] == dn) and r[5] and np.isfinite(r[3]) and np.isfinite(r[4])]
    if len(p) < 5:
        print(f"{(dn or 'ALL'):<12}{len(p):>5}   (too few)")
        continue
    a = np.array([x for x, _ in p]); b = np.array([y for _, y in p])
    print(f"{(dn or 'ALL'):<12}{len(p):>5}{a.mean():>+12.2f}{b.mean():>+12.2f}"
          f"{(a-b).mean():>+10.2f}{sem(a-b):>8.2f}")

# --- 2. selection: sart_full in phantoms with and without victre_fbp -------------
print("\n--- 2. selection test (sart_full, with/without victre_fbp) ---")
print(f"{'density':<12}{'n(with)':>7}{'offset':>9}{'n(w/o)':>8}{'offset':>9}{'diff':>8}")
for dn in DN:
    v = [r[3] for r in recs if r[1] == dn and r[5] and np.isfinite(r[3])]
    y = [r[3] for r in recs if r[1] == dn and not r[5] and np.isfinite(r[3])]
    fk = f"{np.mean(v)-np.mean(y):>+8.2f}" if len(v) >= 5 and len(y) >= 5 else "     ---"
    mv = f"{np.mean(v):>+9.2f}" if len(v) >= 5 else "      ---"
    my = f"{np.mean(y):>+9.2f}" if len(y) >= 5 else "      ---"
    print(f"{dn:<12}{len(v):>7}{mv}{len(y):>8}{my}{fk}")

# --- 3. depth: offset ~ z position -------------------------------------------
print("\n--- 3. depth test (offset = a + slope * z/S) ---")
print(f"{'volume':<12}{'density':<12}{'n':>5}{'intercept':>9}{'slope':>9}{'r':>8}")
for col, name in ((3, "sart_full"), (4, "victre_fbp")):
    for dn in DN:
        p = [(r[2], r[col]) for r in recs if r[1] == dn and np.isfinite(r[col])]
        if len(p) < 10:
            continue
        x = np.array([a for a, _ in p]); y = np.array([b for _, b in p])
        if x.std() < 1e-6:
            continue
        m, k = np.polyfit(x, y, 1)
        r = float(np.corrcoef(x, y)[0, 1])
        print(f"{name:<12}{dn:<12}{len(p):>5}{k:>+9.2f}{m:>+9.2f}{r:>8.2f}")

# --- 4. fatty: phantom by phantom ---------------------------------------------------
print("\n--- 4. fatty phantoms (sart_full, phantom mean) ---")
per = collections.defaultdict(list)
for pid, dn, _, o_s, _, has_vf in recs:
    if dn == "fatty" and np.isfinite(o_s):
        per[(pid, has_vf)].append(o_s)
ms = sorted((np.mean(v), pid, has_vf) for (pid, has_vf), v in per.items())
for mu, pid, has_vf in ms:
    print(f"  {pid:<22}{'vf' if has_vf else '  '}{mu:>+8.2f}")
if ms:
    mus = np.array([m for m, _, _ in ms])
    print(f"  {'phantoms':<22}  {len(mus):>6}   mean {mus.mean():+.2f}"
          f"  sem {sem(mus):.2f}  min {mus.min():+.2f}  max {mus.max():+.2f}")

print("\nDecision: if the difference in 2 is <0.3 AND |slope| in 3 is <0.5 -> DELTA[fatty] is corrected")
print("          and fatty alone is regenerated. Otherwise DELTA is not changed.")

### D4 · Save the pilot output to Drive

`/content/pilot` is on the **VM disk**. If the session drops, 1.9 hours and D3's data are lost.

These 100 phantoms are **bit-identical** to the first 100 of the full generation: the order of `select_patients()`
is deterministic in `rank`, the split comes from `rank` and the frozen manifest, `--limit` affects neither, and the
phantom's RNG is derived from its id. Once moved to Drive, E counts them as finished: 1.9 hours are recovered.

In [ ]:
# the output in SCAN -> Drive. ~31 MB per phantom, minutes through FUSE.
import shutil, time
from pathlib import Path
PILOT = SCAN
n = 0
for s in SPLITS:
    src = PILOT/s
    if not src.is_dir(): continue
    (DRIVE_OUT/s).mkdir(parents=True, exist_ok=True)
    for f in sorted(src.glob("*.npz")):
        dst = DRIVE_OUT/s/f.name
        if dst.exists(): continue
        for attempt in range(3):
            try:
                shutil.copy(str(f), str(dst)); n += 1; break
            except OSError as e:
                print(f"  ! {f.name}: {e} (attempt {attempt+1})"); time.sleep(10)
print(f"{n} phantoms copied to Drive. Total on Drive: {len(drive_done())}")
print(f"NOTE: {SCAN} not deleted -- in case D3 is to be run again.")

## E · Full generation

Resumes where it stopped. If the session drops, run this cell again.
Moves to Drive every `BATCH` phantoms, so a dead session loses at most `BATCH` phantoms.

In [ ]:
import subprocess, sys, time
from pathlib import Path

# Fatty is held back until the D3 decision: if DELTA[fatty] changes, these phantoms were never
# generated; if not, set this to () and continue. No wasted work either way.
DEFER_DENSITIES = ("fatty",)        # set to () after D3

# Phantoms this tab will not take: split by item order so parallel tabs do not duplicate work.
def session_skip():
    if N_SESSIONS <= 1 and not DEFER_DENSITIES: return set(), 0
    sys.path.insert(0, str(NB_DIR))
    import dataset_generate as G
    items = G.select_patients(G.tcia_index(CACHE), G.lesion_index(CACHE))
    skip, deferred = set(), 0
    for i, it in enumerate(items):
        if it["density"] in DEFER_DENSITIES:
            skip.add(it["patient"]); deferred += 1
        elif N_SESSIONS > 1 and i % N_SESSIONS != SESSION:
            skip.add(it["patient"])
    return skip, deferred

SKIP, DEFERRED = session_skip()
print(f"session {SESSION}/{N_SESSIONS}: {len(SKIP)} phantoms skipped "
      f"({DEFERRED} of them in a deferred density: {DEFER_DENSITIES or 'none'})")

round_no = 0
while True:
    round_no += 1
    done = drive_done()
    (LOCAL_OUT/"done.txt").write_text("\n".join(sorted(done | SKIP)))
    before = len(done)
    print(f"\n--- round {round_no} | {before} phantoms on Drive ---", flush=True)

    r = subprocess.run([sys.executable, "dataset_generate.py",
                        "--cache", str(CACHE), "--output", str(LOCAL_OUT),
                        "--done-list", str(LOCAL_OUT/"done.txt"),
                        "--max-new", str(BATCH), "--progress-every", "5"],
                       text=True)
    moved = move_to_drive()
    after = len(drive_done())
    print(f"--- round {round_no}: +{moved} moved, total on Drive {after} ---", flush=True)

    if moved == 0:
        print("\nNo new phantoms generated. Either finished or all failing -> see progress.json.")
        break
    if r.returncode != 0:
        print(f"\nThe generator exited with {r.returncode}; the moved files are safe. Run the cell again.")
        break

## F · Validation

`dataset_test.py`: schema check of every file, calibration of the noise injection, alignment of the second
reference and the baseline reconstructions. **Needs a GPU, ~8 hours.**

Output: `validation.md` + `preview.png`.

In [ ]:
!python dataset_test.py --data {DRIVE_OUT} --metric-splits test

from IPython.display import Markdown, Image, display
display(Markdown((DRIVE_OUT/"validation.md").read_text()))
display(Image(str(DRIVE_OUT/"preview.png")))

### F2 · Expected results

The targets set by the development release. **If one does not hold, its cause is found before the paper is written.**

In [ ]:
import re
txt = (DRIVE_OUT/"validation.md").read_text()
print("Problems: 0            ->", "YES" if "Problems: 0." in txt else "NO  <-- CHECK")
for k, lo, hi in [("proj_50",0.85,1.15),("proj_25",0.85,1.15),("proj_10",0.85,1.15),
                  ("dm_50",0.85,1.15),("dm_25",0.85,1.15),("dm_10",0.85,1.15)]:
    m = re.search(rf"\|\s*{k}\s*\|\s*([\d.]+)", txt)
    if m:
        v = float(m.group(1))
        print(f"noise {k:<8} {v:.3f}    -> {'OK' if lo<=v<=hi else 'OUT OF RANGE  <-- CHECK'}")
m = re.search(r"SART 25% dose[\s\S]*?\|\s*test\s*\|\s*all\s*\|\s*\d+\s*\|\s*([\d.]+)", txt)
if m: print(f"SART 25 % PSNR {float(m.group(1)):.2f} dB  -> DBT 2026v1: 27.83 (should be close)")
m = re.search(r"corr\(victre_fbp, sart_full\)[^|]*\|\s*([\d.]+)", txt)
if m: print(f"second-reference alignment {float(m.group(1)):.3f}  -> should be above 0.30")

## G · Manifest — over the whole collection

In [ ]:
# The manifest is built in one pass. Fast: it does NOT open the large arrays, it reads only their headers.
# Resumes where it stopped (intermediate record on Drive: manifest_parts.jsonl), reports progress every 100 phantoms.
import numpy as np, csv, json, zipfile, time, collections
from pathlib import Path

def npy_shape(zf, name):
    with zf.open(name + ".npy") as h:
        v = np.lib.format.read_magic(h)
        rd = np.lib.format.read_array_header_1_0 if v == (1, 0) else np.lib.format.read_array_header_2_0
        shape, _, _ = rd(h)
    return shape

def small(zf, name):
    with zf.open(name + ".npy") as h:
        return np.lib.format.read_array(h)

PARTS = DRIVE_OUT/"manifest_parts.jsonl"
seen = {}
if PARTS.exists():
    for ln in PARTS.read_text().splitlines():
        if ln.strip():
            r = json.loads(ln); seen[r["patient"]] = r
files = [(s, f) for s in SPLITS for f in sorted((DRIVE_OUT/s).glob("*.npz")) if ".partial" not in f.name]
print(f"{len(files)} files, {len(seen)} read before")

t0, buf = time.time(), []
todo = [(s, f) for s, f in files if f.stem not in seen]
for i, (s, f) in enumerate(todo, 1):
    with zipfile.ZipFile(f) as zf:
        les = small(zf, "lesions")
        r = {"patient": f.stem, "density": str(small(zf, "density")),
             "signal_present": bool(small(zf, "signal_present")), "split": s,
             "slices": int(npy_shape(zf, "sart_full")[0]),
             "lesions": int((les[:, 4] >= 0).sum()) if les.size else 0,
             "photons_per_pixel": round(float(small(zf, "photons_per_pixel")), 1),
             "electronic_sigma": round(float(small(zf, "electronic_sigma")), 2),
             "victre_fbp": int(bool(small(zf, "victre_fbp_present"))),
             "victre_fbp_saturated": round(float(small(zf, "victre_fbp_saturated")), 6),
             "size_mb": round(f.stat().st_size / 1e6, 1)}
    seen[r["patient"]] = r; buf.append(r)
    if len(buf) >= 100 or i == len(todo):
        with open(PARTS, "a") as h:
            h.writelines(json.dumps(x) + "\n" for x in buf)
        buf = []
        print(f"  {i}/{len(todo)}   {(time.time()-t0)/i:.2f} s/phantom", flush=True)

rows = [seen[f.stem] for s, f in files]
with open(DRIVE_OUT/"manifest.csv", "w", newline="") as h:
    w = csv.DictWriter(h, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

print(f"\n{len(rows)} phantoms, {sum(r['slices'] for r in rows)} slices, "
      f"{sum(r['size_mb'] for r in rows)/1000:.1f} GB")
print("density:", dict(collections.Counter(r['density'] for r in rows)))
print("split  :", dict(collections.Counter(r['split'] for r in rows)))
print("signal :", dict(collections.Counter(r['signal_present'] for r in rows)))
print("with victre_fbp:", sum(r['victre_fbp'] for r in rows))
print("signal-present phantoms without lesions:", sum(1 for r in rows if r['signal_present'] and r['lesions'] == 0))

## H · Status — at any time

In [ ]:
import json, collections
from pathlib import Path
done = drive_done()
print(f"finished on Drive: {len(done)} / 2993   ({100*len(done)/2993:.1f} %)")
p = DRIVE_OUT/"progress.json"
if p.exists():
    pr = json.loads(p.read_text())
    print(f"last update     : {pr.get('updated')}")
    print(f"per phantom     : {pr.get('minutes_per_patient')} min")
    print(f"estimated left  : {pr.get('eta_hours')} hours")
    if pr.get("failed"):
        print(f"failed          : {len(pr['failed'])}")
        for k,v in list(pr["failed"].items())[:5]: print(f"   {k}: {v}")